# exp6 R1 4 ablation training

**Self-contained Colab notebook.** Run the setup cells once (top to bottom), then the experiment. The four shared modules are written to `/content` automatically, so you do NOT need to upload a `shared/` folder. Edit Drive paths only in the `config_paths.py` cell below.

> Runtime: set **Runtime -> Change runtime type -> GPU** for the inference and training experiments.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
print('Drive mounted; /content on path.')

In [ ]:
# scikit-image backs metrics.calc_ssim; guarantee it for every notebook
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-image'])
print('scikit-image ready')

In [ ]:
%%writefile /content/config_paths.py
"""
Central path + constant configuration for all JBHI revision Colab experiments.

Derived verbatim from the author's existing notebooks:
  - COLAB_IEEE_FIGURES.ipynb      (paper baselines + EPHNOGRAM chunks + split)
  - cross_dataset_validation.ipynb (SensSmart-Tech + Trail-4 generator)

Every experiment script imports from this file so paths live in ONE place.
If your Drive layout differs, edit ONLY this file.
"""

import os
import torch

# ============================================================================
# >>> EDIT THIS BLOCK IF YOUR DRIVE MOVES <<<
# Updated 2026-07-14: the dataset is now a SHARED folder mounted at
# /content/drive/MyDrive/JBHI-Revision/October Thesis Dataset. If it moves again,
# change ONLY DRIVE_ROOT and OCT_DIR. NOTE: DATA is READ from OCT_DIR; everything
# the code WRITES (retrained checkpoints, outputs) goes under your own Drive
# (OUT_ROOT), so a read-only shared dataset still works.
# ============================================================================
DRIVE_ROOT = "/content/drive/MyDrive"
# Shared dataset root: contains "Refine-Chunked", "Best-epoch-trail-4", etc. (read-only OK).
OCT_DIR    = f"{DRIVE_ROOT}/JBHI-Revision/October Thesis Dataset"

# ----------------------------------------------------------------------------
# EPHNOGRAM data (training-domain test set = 453 segments, 3 subjects)
# .npz keys per chunk: 'ecg', 'pcg', 'subject_id', 'scenario'
#   ecg, pcg : float32 arrays, 64000 samples (8 s @ 8 kHz), RAW (not normalized)
#   subject_id : e.g. 'S009', 'S019', 'S023'
#   scenario   : free-text, e.g. 'rest', 'exercise bicycle stress test'
# ----------------------------------------------------------------------------
EPH_CHUNKS_DIR = f"{OCT_DIR}/Refine-Chunked"
# Subject split. keys: train/val/test_subjects. IF THIS FILE IS ABSENT the
# notebooks fall back to an arbitrary split, which gives the WRONG test set
# (the 2026-07-10 run hit this: 22 segments / 4 subjects instead of the paper's
# 453 segments / 3 subjects). Upload the real split JSON, or point this at it.
SPLIT_FILE     = f"{EPH_CHUNKS_DIR}/train_val_test_split.json"

# ----------------------------------------------------------------------------
# Model checkpoints  --  CONFIRM ALL FOUR EXIST IN THE NEW ACCOUNT
#
#   "Proposed" MUST be the Trail-4 generator (polarity fix, BatchNorm output),
#   i.e. trail4_best.pt or trail4_generator.pth. A "trial2_*" / Trail-2 file is
#   the WRONG (Tanh) model: it loads only partially under strict=False and
#   produces a near-flat, invalid reconstruction (see the 2026-07-10 run).
#
#   The three baselines lived under a SEPARATE top-level folder
#   (Paper1_ECG_Reconstruction) in the old account. Confirm they were migrated;
#   if they now sit under OCT_DIR, repoint them below.
# ----------------------------------------------------------------------------
CKPT = {
    # Baselines are RETRAINED by exp7 and SAVED to your own writable Drive
    # (DRIVE_ROOT/JBHI_Revision/baseline_checkpoints == OUT_ROOT/baseline_checkpoints),
    # so a read-only shared dataset is fine. Same paths are used to load them later.
    "CNN":      f"{DRIVE_ROOT}/JBHI_Revision/baseline_checkpoints/CNN_EncoderDecoder/best_model.pt",
    "UNet":     f"{DRIVE_ROOT}/JBHI_Revision/baseline_checkpoints/UNet_Baseline/best_model.pt",
    "WGAN-GP":  f"{DRIVE_ROOT}/JBHI_Revision/baseline_checkpoints/WGAN_GP/best_model.pt",
    "Proposed": f"{OCT_DIR}/Best-epoch-trail-4/trail4_best.pt",   # read from the shared dataset
}
CKPT_PROPOSED_ALT = f"{OCT_DIR}/trail4_generator.pth"   # fallback copy (state_dict)

# ----------------------------------------------------------------------------
# SensSmart-Tech cross-dataset (32 subjects, 338 records @ 1 kHz -> 8 kHz)
#   WFDB pairs: {base}_ecg.hea/.dat and {base}_pcg.hea/.dat ; ECG lead II
# ----------------------------------------------------------------------------
SENSSMART_WFDB  = f"{OCT_DIR}/senssmarttech-database/WFDB"
SENSSMART_PREP  = f"{OCT_DIR}/milestone2_preprocessed"     # cached .npz (keys ecg/pcg)

# ----------------------------------------------------------------------------
# Output root for ALL revision artifacts (created on demand)
#   Each experiment writes to a named sub-folder here.
# ----------------------------------------------------------------------------
OUT_ROOT = f"{DRIVE_ROOT}/JBHI_Revision"

def out_dir(name):
    """Return (and create) an output sub-folder under OUT_ROOT."""
    p = os.path.join(OUT_ROOT, name)
    os.makedirs(p, exist_ok=True)
    return p

# ----------------------------------------------------------------------------
# Signal + inference constants
# ----------------------------------------------------------------------------
FS          = 8000        # sampling rate (Hz)
SEG_LEN     = 64000       # samples per segment (8 s)
NUM_SCENARIOS = 3         # rest / exercise / recovery one-hot
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# R-peak detector settings (identical to metrics.ipynb; keep fixed for reproducibility)
RPEAK_DISTANCE   = 400    # min inter-peak samples = 50 ms @ 8 kHz
RPEAK_PROMINENCE = 0.5

# SensSmart preprocessing bands (from cross_dataset_validation.ipynb)
ECG_BAND = (0.5, 40.0)
PCG_BAND = (25.0, 400.0)
FILTER_ORDER = 4

# Scenario one-hots
SCEN_REST     = [1.0, 0.0, 0.0]
SCEN_EXERCISE = [0.0, 1.0, 0.0]
SCEN_RECOVERY = [0.0, 0.0, 1.0]


def scenario_to_index(scenario_str):
    """Map EPHNOGRAM free-text scenario -> class index {0,1,2}.

    Matches the mapping used in COLAB_IEEE_FIGURES.ipynb.
    """
    s = str(scenario_str).lower()
    if "rest" in s or "laying" in s:
        return 0
    if "exercise" in s or "bicycle" in s or "treadmill" in s or "running" in s:
        return 1
    return 2

In [ ]:
%%writefile /content/models.py
"""
Model architectures for the JBHI revision experiments.

Reproduced VERBATIM from the author's COLAB_IEEE_FIGURES.ipynb (paper Table II
baselines) and models.ipynb (ablation trails), so the checkpoints in
config_paths.CKPT load cleanly.

Which class loads which checkpoint
----------------------------------
    CKPT['CNN']      -> CNNEncoderDecoder()        # no scenario, forward(x)
    CKPT['UNet']     -> UNetGenerator()            # pix2pix, forward(x)
    CKPT['WGAN-GP']  -> UNetGenerator()            # same generator as U-Net
    CKPT['Proposed'] -> ImprovedGeneratorNoTanh()  # scenario, forward(x, s_onehot)

The proposed model's forward takes a scenario one-hot; the three baselines take
only the PCG. Every inference script must branch on this (helper `run_generator`
below does it for you).

For the R1.4 component ablation, use `ConfigurableGenerator`, which exposes the
three grouped factors as switches: use_attention, use_dilated, and
output_activation in {'batchnorm','tanh','linear'}.

NOTE ON PARAM COUNTS: the submitted paper's numbers (CNN 9.8M, U-Net 13.6M,
Proposed 14.0M) do NOT match these classes (which give ~larger counts). Always
report count_params(model_loaded_from_checkpoint) computed here -- see R2.2/R2.5.
"""

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ===========================================================================
# Shared bottleneck modules
# ===========================================================================
class MultiHeadSelfAttention(nn.Module):
    """8-head self-attention over the temporal axis at the bottleneck."""
    def __init__(self, channels, num_heads=8, dropout=0.1):
        super().__init__()
        assert channels % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Conv1d(channels, channels * 3, kernel_size=1, bias=False)
        self.proj = nn.Conv1d(channels, channels, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, C, T = x.shape
        qkv = self.qkv(x).reshape(B, 3, self.num_heads, self.head_dim, T)
        qkv = qkv.permute(1, 0, 2, 4, 3)          # (3, B, heads, T, head_dim)
        # .contiguous() is REQUIRED for SDPA to select the flash / memory-efficient
        # kernel (O(T) memory). Without it the permuted views fall back to the MATH
        # kernel, which materializes the T x T score matrix -- that OOM-ed the
        # attention variants even on a 40 GB A100 at batch 48. Mathematically
        # identical to softmax(QK^T/sqrt(d))V, so existing checkpoints are unaffected.
        q, k, v = qkv[0].contiguous(), qkv[1].contiguous(), qkv[2].contiguous()
        p = self.dropout.p if self.training else 0.0
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=p)
        out = out.transpose(2, 3).reshape(B, C, T)
        return self.proj(out) + x


class MultiScaleDilatedBlock(nn.Module):
    """Parallel dilated convs at rates 1, 2, 4 with a 1x1 fusion + residual."""
    def __init__(self, channels):
        super().__init__()
        c1 = channels // 3 + (1 if channels % 3 >= 1 else 0)
        c2 = channels // 3 + (1 if channels % 3 >= 2 else 0)
        c3 = channels // 3
        self.conv_d1 = nn.Conv1d(channels, c1, 3, padding=1, dilation=1)
        self.conv_d2 = nn.Conv1d(channels, c2, 3, padding=2, dilation=2)
        self.conv_d4 = nn.Conv1d(channels, c3, 3, padding=4, dilation=4)
        self.fusion = nn.Sequential(
            nn.Conv1d(channels, channels, 1),
            nn.BatchNorm1d(channels), nn.ReLU(inplace=True))

    def forward(self, x):
        cat = torch.cat([self.conv_d1(x), self.conv_d2(x), self.conv_d4(x)], dim=1)
        return self.fusion(cat) + x


class EnhancedResidualBlock(nn.Module):
    """Residual block with optional MHSA and MSDC (Trails 2-4)."""
    def __init__(self, channels, use_attention=False, use_dilated=True):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm1d(channels)
        self.use_dilated = use_dilated
        if use_dilated:
            self.dilated_block = MultiScaleDilatedBlock(channels)
        self.conv2 = nn.Conv1d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(channels)
        self.use_attention = use_attention
        if use_attention:
            self.attention = MultiHeadSelfAttention(channels, num_heads=8)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        if self.use_dilated:
            out = self.dilated_block(out)
        out = self.bn2(self.conv2(out))
        if self.use_attention:
            out = self.attention(out)
        return self.relu(out + residual)


# ===========================================================================
# Paper Table II baseline: CNN Encoder-Decoder (8-level, no skips, no scenario)
# ===========================================================================
class ConvBlock(nn.Module):
    def __init__(self, i, o, k=4, s=2, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(i, o, k, s, p, bias=False),
            nn.BatchNorm1d(o), nn.LeakyReLU(0.2, inplace=True))
    def forward(self, x): return self.block(x)


class DeconvBlock(nn.Module):
    def __init__(self, i, o, k=4, s=2, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose1d(i, o, k, s, p, bias=False),
            nn.BatchNorm1d(o), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)


class CNNEncoderDecoder(nn.Module):
    """Paper CNN baseline -> loads CKPT['CNN']. forward(x)."""
    def __init__(self, in_channels=1, out_channels=1, base_filters=64):
        super().__init__()
        bf = base_filters
        self.enc1 = ConvBlock(in_channels, bf)
        self.enc2 = ConvBlock(bf, bf * 2)
        self.enc3 = ConvBlock(bf * 2, bf * 4)
        self.enc4 = ConvBlock(bf * 4, bf * 8)
        self.enc5 = ConvBlock(bf * 8, bf * 8)
        self.enc6 = ConvBlock(bf * 8, bf * 8)
        self.enc7 = ConvBlock(bf * 8, bf * 8)
        self.enc8 = ConvBlock(bf * 8, bf * 8)
        self.dec1 = DeconvBlock(bf * 8, bf * 8)
        self.dec2 = DeconvBlock(bf * 8, bf * 8)
        self.dec3 = DeconvBlock(bf * 8, bf * 8)
        self.dec4 = DeconvBlock(bf * 8, bf * 8)
        self.dec5 = DeconvBlock(bf * 8, bf * 4)
        self.dec6 = DeconvBlock(bf * 4, bf * 2)
        self.dec7 = DeconvBlock(bf * 2, bf)
        self.final = nn.ConvTranspose1d(bf, out_channels, 4, 2, 1)

    def forward(self, x):
        n = x.size(2)
        e = self.enc1(x); e = self.enc2(e); e = self.enc3(e); e = self.enc4(e)
        e = self.enc5(e); e = self.enc6(e); e = self.enc7(e); e = self.enc8(e)
        d = self.dec1(e); d = self.dec2(d); d = self.dec3(d); d = self.dec4(d)
        d = self.dec5(d); d = self.dec6(d); d = self.dec7(d)
        out = self.final(d)
        if out.size(2) != n:
            diff = n - out.size(2)
            out = F.pad(out, (diff // 2, diff - diff // 2))
        return out


# ===========================================================================
# Paper Table II baseline: pix2pix U-Net (shared by U-Net and WGAN-GP)
# ===========================================================================
class UNetBlock(nn.Module):
    def __init__(self, i, o, normalize=True, dropout=0.0):
        super().__init__()
        layers = [nn.Conv1d(i, o, 4, 2, 1, bias=False)]
        if normalize:
            layers.append(nn.BatchNorm1d(o))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)


class UNetUpBlock(nn.Module):
    def __init__(self, i, o, dropout=0.0):
        super().__init__()
        layers = [nn.ConvTranspose1d(i, o, 4, 2, 1, bias=False),
                  nn.BatchNorm1d(o), nn.ReLU(inplace=True)]
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x, skip):
        x = self.block(x)
        if x.size(2) != skip.size(2):
            diff = skip.size(2) - x.size(2)
            x = F.pad(x, (diff // 2, diff - diff // 2))
        return torch.cat([x, skip], dim=1)


class UNetGenerator(nn.Module):
    """Paper U-Net / WGAN-GP generator -> loads CKPT['UNet'] or CKPT['WGAN-GP']. forward(x)."""
    def __init__(self, in_channels=1, out_channels=1, features=64):
        super().__init__()
        f = features
        self.e1 = UNetBlock(in_channels, f, normalize=False)
        self.e2 = UNetBlock(f, f * 2); self.e3 = UNetBlock(f * 2, f * 4)
        self.e4 = UNetBlock(f * 4, f * 8); self.e5 = UNetBlock(f * 8, f * 8)
        self.e6 = UNetBlock(f * 8, f * 8); self.e7 = UNetBlock(f * 8, f * 8)
        self.e8 = UNetBlock(f * 8, f * 8, normalize=False)
        self.d1 = UNetUpBlock(f * 8, f * 8, dropout=0.5)
        self.d2 = UNetUpBlock(f * 16, f * 8, dropout=0.5)
        self.d3 = UNetUpBlock(f * 16, f * 8, dropout=0.5)
        self.d4 = UNetUpBlock(f * 16, f * 8)
        self.d5 = UNetUpBlock(f * 16, f * 4); self.d6 = UNetUpBlock(f * 8, f * 2)
        self.d7 = UNetUpBlock(f * 4, f)
        self.final = nn.ConvTranspose1d(f * 2, out_channels, 4, 2, 1)

    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(e1); e3 = self.e3(e2); e4 = self.e4(e3)
        e5 = self.e5(e4); e6 = self.e6(e5); e7 = self.e7(e6); e8 = self.e8(e7)
        d = self.d1(e8, e7); d = self.d2(d, e6); d = self.d3(d, e5); d = self.d4(d, e4)
        d = self.d5(d, e3); d = self.d6(d, e2); d = self.d7(d, e1)
        out = self.final(d)
        if out.size(2) != x.size(2):
            diff = x.size(2) - out.size(2)
            out = F.pad(out, (diff // 2, diff - diff // 2))
        return out


# ===========================================================================
# Proposed generator (Trail-4): U-Net + MHSA + MSDC + scenario, BatchNorm output
# ===========================================================================
class ImprovedGeneratorNoTanh(nn.Module):
    """Proposed model -> loads CKPT['Proposed']. forward(x, scenario_onehot)."""
    def __init__(self, signal_length=64000, base_channels=64, num_scenarios=3):
        super().__init__()
        bc = base_channels
        self.scenario_embed = nn.Linear(num_scenarios, bc)
        self.enc1 = nn.Sequential(nn.Conv1d(1, bc, 7, 2, 3), nn.BatchNorm1d(bc), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv1d(bc, bc*2, 5, 2, 2), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.enc3 = nn.Sequential(nn.Conv1d(bc*2, bc*4, 5, 2, 2), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.enc4 = nn.Sequential(nn.Conv1d(bc*4, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.enc5 = nn.Sequential(nn.Conv1d(bc*8, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.bottleneck = nn.Sequential(
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True),
            EnhancedResidualBlock(bc*8, use_attention=True, use_dilated=True))
        self.dec5 = nn.Sequential(nn.ConvTranspose1d(bc*8, bc*8, 4, 2, 1), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.dec4 = nn.Sequential(nn.Conv1d(bc*16, bc*8, 1), nn.ConvTranspose1d(bc*8, bc*4, 4, 2, 1), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.dec3 = nn.Sequential(nn.Conv1d(bc*8, bc*4, 1), nn.ConvTranspose1d(bc*4, bc*2, 4, 2, 1), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.dec2 = nn.Sequential(nn.Conv1d(bc*4, bc*2, 1), nn.ConvTranspose1d(bc*2, bc, 4, 2, 1), nn.BatchNorm1d(bc), nn.ReLU(True))
        self.dec1 = nn.Sequential(nn.Conv1d(bc*2, bc, 1), nn.ConvTranspose1d(bc, 1, 4, 2, 1), nn.BatchNorm1d(1))

    def forward(self, x, scenario_onehot):
        s = self.scenario_embed(scenario_onehot).unsqueeze(2)
        e1 = self.enc1(x) + s
        e2 = self.enc2(e1); e3 = self.enc3(e2); e4 = self.enc4(e3); e5 = self.enc5(e4)
        b = self.bottleneck(e5)
        d5 = torch.cat([self.dec5(b), e4], dim=1)
        d4 = torch.cat([self.dec4(d5), e3], dim=1)
        d3 = torch.cat([self.dec3(d4), e2], dim=1)
        d2 = torch.cat([self.dec2(d3), e1], dim=1)
        return self.dec1(d2)


# ===========================================================================
# Configurable generator for the R1.4 component ablation
# ===========================================================================
class ConfigurableGenerator(nn.Module):
    """Ablation generator: toggle each grouped factor independently.

    use_attention : add MHSA in the 4 bottleneck blocks
    use_dilated   : add MSDC in the 4 bottleneck blocks
    output_activation : 'batchnorm' (proposed) | 'tanh' | 'linear'
    use_scenario  : condition on the scenario one-hot (keep True to match proposed)

    Ablation rows the reviewer asked for map to:
        U-Net only          -> use_attention=False, use_dilated=False
        U-Net + MSDC        -> use_attention=False, use_dilated=True
        U-Net + MHSA        -> use_attention=True,  use_dilated=False
        U-Net + both        -> use_attention=True,  use_dilated=True
        Proposed + Tanh     -> both True, output_activation='tanh'
        Proposed + linear   -> both True, output_activation='linear'
        Proposed (BatchNorm)-> both True, output_activation='batchnorm'
    Polarity-on/off is a DATA-loader switch; adversarial-on/off is a TRAINING
    switch -- neither is an architecture change, so control them in the trainer.
    """
    def __init__(self, base_channels=64, num_scenarios=3,
                 use_attention=True, use_dilated=True,
                 output_activation="batchnorm", use_scenario=True):
        super().__init__()
        bc = base_channels
        self.use_scenario = use_scenario
        if use_scenario:
            self.scenario_embed = nn.Linear(num_scenarios, bc)
        self.enc1 = nn.Sequential(nn.Conv1d(1, bc, 7, 2, 3), nn.BatchNorm1d(bc), nn.ReLU(True))
        self.enc2 = nn.Sequential(nn.Conv1d(bc, bc*2, 5, 2, 2), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.enc3 = nn.Sequential(nn.Conv1d(bc*2, bc*4, 5, 2, 2), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.enc4 = nn.Sequential(nn.Conv1d(bc*4, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.enc5 = nn.Sequential(nn.Conv1d(bc*8, bc*8, 5, 2, 2), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.bottleneck = nn.Sequential(*[
            EnhancedResidualBlock(bc*8, use_attention=use_attention, use_dilated=use_dilated)
            for _ in range(4)])
        self.dec5 = nn.Sequential(nn.ConvTranspose1d(bc*8, bc*8, 4, 2, 1), nn.BatchNorm1d(bc*8), nn.ReLU(True))
        self.dec4 = nn.Sequential(nn.Conv1d(bc*16, bc*8, 1), nn.ConvTranspose1d(bc*8, bc*4, 4, 2, 1), nn.BatchNorm1d(bc*4), nn.ReLU(True))
        self.dec3 = nn.Sequential(nn.Conv1d(bc*8, bc*4, 1), nn.ConvTranspose1d(bc*4, bc*2, 4, 2, 1), nn.BatchNorm1d(bc*2), nn.ReLU(True))
        self.dec2 = nn.Sequential(nn.Conv1d(bc*4, bc*2, 1), nn.ConvTranspose1d(bc*2, bc, 4, 2, 1), nn.BatchNorm1d(bc), nn.ReLU(True))
        head = [nn.Conv1d(bc*2, bc, 1), nn.ConvTranspose1d(bc, 1, 4, 2, 1)]
        if output_activation == "batchnorm":
            head.append(nn.BatchNorm1d(1))
        elif output_activation == "tanh":
            head.append(nn.Tanh())
        elif output_activation == "linear":
            pass
        else:
            raise ValueError(f"unknown output_activation {output_activation}")
        self.dec1 = nn.Sequential(*head)

    def forward(self, x, scenario_onehot=None):
        e1 = self.enc1(x)
        if self.use_scenario and scenario_onehot is not None:
            e1 = e1 + self.scenario_embed(scenario_onehot).unsqueeze(2)
        e2 = self.enc2(e1); e3 = self.enc3(e2); e4 = self.enc4(e3); e5 = self.enc5(e4)
        b = self.bottleneck(e5)
        d5 = torch.cat([self.dec5(b), e4], dim=1)
        d4 = torch.cat([self.dec4(d5), e3], dim=1)
        d3 = torch.cat([self.dec3(d4), e2], dim=1)
        d2 = torch.cat([self.dec2(d3), e1], dim=1)
        return self.dec1(d2)


# ===========================================================================
# Discriminator (for R1.4 adversarial retraining)
# ===========================================================================
class SpectralNormDiscriminator(nn.Module):
    """Spectrally-normalized PatchGAN critic with scenario conditioning."""
    def __init__(self, base_channels=64, num_scenarios=3):
        super().__init__()
        bc = base_channels
        self.scenario_embed = nn.Linear(num_scenarios, bc)
        sn = nn.utils.spectral_norm
        self.conv1 = sn(nn.Conv1d(2, bc, 7, 2, 3))
        self.conv2 = sn(nn.Conv1d(bc, bc*2, 5, 2, 2)); self.bn2 = nn.BatchNorm1d(bc*2)
        self.conv3 = sn(nn.Conv1d(bc*2, bc*4, 5, 2, 2)); self.bn3 = nn.BatchNorm1d(bc*4)
        self.conv4 = sn(nn.Conv1d(bc*4, bc*8, 5, 2, 2)); self.bn4 = nn.BatchNorm1d(bc*8)
        self.conv5 = sn(nn.Conv1d(bc*8, bc*8, 5, 2, 2)); self.bn5 = nn.BatchNorm1d(bc*8)
        self.pool = nn.AdaptiveAvgPool1d(1); self.fc = nn.Linear(bc*8, 1)
        self.lrelu = nn.LeakyReLU(0.2, True)

    def forward(self, pcg, ecg, scenario_onehot):
        x = torch.cat([pcg, ecg], dim=1)
        s = self.scenario_embed(scenario_onehot).unsqueeze(2)
        x = self.lrelu(self.conv1(x)) + s
        x = self.lrelu(self.bn2(self.conv2(x)))
        x = self.lrelu(self.bn3(self.conv3(x)))
        x = self.lrelu(self.bn4(self.conv4(x)))
        x = self.lrelu(self.bn5(self.conv5(x)))
        return torch.sigmoid(self.fc(self.pool(x).squeeze(2)))


# ===========================================================================
# WGAN-GP critic (for retraining the WGAN-GP baseline; extracted verbatim from
# the original 05_GAN_WGAN_GP notebook). No BatchNorm, no sigmoid -> a valid
# WGAN critic. The generator is the plain UNetGenerator above.
# ===========================================================================
class WGANCritic(nn.Module):
    def __init__(self, in_channels=2, features=64):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv1d(in_channels, features, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features, features * 2, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features * 2, features * 4, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features * 4, features * 8, 4, 1, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(features * 8, 1, 4, 1, 1),
        )

    def forward(self, x, y):
        return self.model(torch.cat([x, y], dim=1))


def compute_gradient_penalty(critic, pcg, real_ecg, fake_ecg, device):
    """WGAN-GP gradient penalty. Interpolates the ECG only; PCG conditions."""
    bs = real_ecg.size(0)
    alpha = torch.rand(bs, 1, 1, device=device)
    interp = (alpha * real_ecg + (1 - alpha) * fake_ecg).requires_grad_(True)
    c_interp = critic(pcg, interp)
    grads = torch.autograd.grad(
        outputs=c_interp, inputs=interp,
        grad_outputs=torch.ones_like(c_interp),
        create_graph=True, retain_graph=True)[0]
    grads = grads.view(bs, -1)
    return ((grads.norm(2, dim=1) - 1) ** 2).mean()


# ===========================================================================
# Loading + utility helpers
# ===========================================================================
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def load_model(model, checkpoint_path, device="cpu", strict=False):
    """Load a checkpoint into `model`, handling common save formats."""
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    if isinstance(ckpt, dict):
        for key in ("generator_state_dict", "model_state_dict", "state_dict"):
            if key in ckpt:
                ckpt = ckpt[key]; break
    state = {k.replace("module.", ""): v for k, v in ckpt.items()} \
            if isinstance(ckpt, dict) else ckpt
    result = model.load_state_dict(state, strict=strict)
    # Loud guard: a large key mismatch means the checkpoint does not fit this
    # architecture (e.g. a Trail-2/trial2 file loaded into the Trail-4 model).
    n_model = max(len(model.state_dict()), 1)
    n_missing, n_unexpected = len(result.missing_keys), len(result.unexpected_keys)
    tag = os.path.basename(checkpoint_path)
    if n_missing or n_unexpected:
        flag = "  <-- LIKELY WRONG CHECKPOINT FOR THIS MODEL" if n_missing > 0.10 * n_model else ""
        print(f"[load_model] {tag}: {n_missing}/{n_model} params missing, "
              f"{n_unexpected} unexpected{flag}")
    else:
        print(f"[load_model] {tag}: exact match ({n_model} tensors).")
    model.to(device).eval()
    return model, result


def build_paper_models(ckpt_dict, device="cpu"):
    """Load all four paper models. Returns {name: model}."""
    models = {}
    models["CNN"], _     = load_model(CNNEncoderDecoder(), ckpt_dict["CNN"], device)
    models["UNet"], _    = load_model(UNetGenerator(), ckpt_dict["UNet"], device)
    models["WGAN-GP"], _ = load_model(UNetGenerator(), ckpt_dict["WGAN-GP"], device)
    models["Proposed"], _= load_model(ImprovedGeneratorNoTanh(), ckpt_dict["Proposed"], device)
    return models


def run_generator(model, pcg_tensor, scenario_tensor, is_proposed):
    """Uniform inference call. `is_proposed` selects the (x, s) signature."""
    with torch.no_grad():
        if is_proposed:
            out = model(pcg_tensor, scenario_tensor)
        else:
            out = model(pcg_tensor)
    return out.squeeze().cpu().numpy()

In [ ]:
%%writefile /content/metrics.py
"""
Metrics for the JBHI revision.

Two families:
  (A) Signal-fidelity metrics  -- identical to the submitted paper's metrics.ipynb
                                  (PSNR, SNR, SSIM, Pearson r, MAE, RMSE,
                                   R-peak AMPLITUDE error, QRS-width error).
  (B) Temporal R-peak DETECTION metrics  -- NEW, added to answer Reviewer 1 #1.
       These detect R-peaks INDEPENDENTLY in the reconstructed ECG and match
       them to ground-truth peaks within a tolerance window, then report
       sensitivity, PPV, F1, FP, FN, timing-error stats, RR-interval error,
       and heart-rate error. This is the metric the paper was missing.

Keep the detector settings fixed (config_paths.RPEAK_*) so every experiment and
the paper agree.
"""

import numpy as np
import scipy.signal as signal
from scipy.stats import pearsonr


# ---------------------------------------------------------------------------
# R-peak detection (single, shared definition)
# ---------------------------------------------------------------------------
def detect_r_peaks(ecg, fs=8000, distance=400, prominence=0.5):
    """scipy find_peaks on an ECG trace. Returns integer sample indices."""
    peaks, _ = signal.find_peaks(np.asarray(ecg).ravel(),
                                 distance=distance, prominence=prominence)
    return peaks


# ---------------------------------------------------------------------------
# (A) Signal-fidelity metrics
# ---------------------------------------------------------------------------
def calc_psnr(real, fake):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    mse = np.mean((real - fake) ** 2)
    if mse < 1e-10:
        return 100.0
    peak = np.max(np.abs(real))
    return float(20 * np.log10(peak / np.sqrt(mse)))


def calc_snr(real, fake):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    noise = real - fake
    return float(10 * np.log10(np.sum(real ** 2) / (np.sum(noise ** 2) + 1e-12)))


def calc_correlation(real, fake):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    try:
        c, _ = pearsonr(real, fake)
        return float(c)
    except Exception:
        return 0.0


def calc_ssim(real, fake):
    from skimage.metrics import structural_similarity as compare_ssim  # lazy: Colab-only dep
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    r = (real - real.min()) / (real.max() - real.min() + 1e-8)
    f = (fake - fake.min()) / (fake.max() - fake.min() + 1e-8)
    return float(compare_ssim(r, f, data_range=1.0))


def calc_mae(real, fake):
    return float(np.mean(np.abs(np.asarray(real).ravel() - np.asarray(fake).ravel())))


def calc_rmse(real, fake):
    d = np.asarray(real).ravel() - np.asarray(fake).ravel()
    return float(np.sqrt(np.mean(d ** 2)))


def calc_r_peak_amplitude_error(real, fake, fs=8000, normalize01=True, **kw):
    """Paper's 'R-peak error' -> renamed 'R-peak AMPLITUDE error'.

    Detects R-peaks on the GROUND TRUTH only, then averages the amplitude error at
    those locations. Kept ONLY as a fidelity measure; NOT a timing metric (R1 #1).

    normalize01=True (DEFAULT) min-max normalizes both signals to [0,1] before the
    error, which REPRODUCES the paper's ~0.126 scale (confirmed: the paper's metric
    is [0,1]-normalized). normalize01=False gives the raw z-scored form, which is
    ~8x larger (z-scored ECG spans ~6-8 units vs 1 for [0,1]) -- that is what the
    first exp4/6/7 runs reported (~1.0). Use the default for consistency with the
    manuscript. Returns np.nan when the ground truth has no detectable R-peaks.
    """
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    peaks = detect_r_peaks(real, fs=fs, **kw)
    if len(peaks) == 0:
        return np.nan
    if normalize01:
        real = (real - real.min()) / (real.max() - real.min() + 1e-8)
        fake = (fake - fake.min()) / (fake.max() - fake.min() + 1e-8)
    return float(np.mean(np.abs(real[peaks] - fake[peaks])))


def _qrs_width_at_peak(ecg, peak_idx, fs=8000, threshold=0.5, search_ms=25):
    amp = abs(ecg[peak_idx]); thr = threshold * amp
    win = int(search_ms / 1000 * fs)
    q = peak_idx
    for j in range(peak_idx - 1, max(peak_idx - win, 0), -1):
        if abs(ecg[j]) < thr:
            q = j; break
    s = peak_idx
    for j in range(peak_idx + 1, min(peak_idx + win, len(ecg))):
        if abs(ecg[j]) < thr:
            s = j; break
    return (s - q) / fs * 1000.0


def calc_qrs_width_error(real, fake, fs=8000, **kw):
    real, fake = np.asarray(real).ravel(), np.asarray(fake).ravel()
    peaks = detect_r_peaks(real, fs=fs, **kw)
    if len(peaks) == 0:
        return np.nan
    errs = [abs(_qrs_width_at_peak(real, p, fs) - _qrs_width_at_peak(fake, p, fs))
            for p in peaks]
    return float(np.mean(errs))


# ---------------------------------------------------------------------------
# (B) Temporal R-peak DETECTION metrics  (NEW -- Reviewer 1 #1)
# ---------------------------------------------------------------------------
def match_peaks(true_peaks, pred_peaks, tol_samples):
    """Greedy nearest-neighbour matching within a tolerance window.

    Each true peak claims the closest unused predicted peak within tol_samples.
    Returns (matches, TP, FP, FN, signed_timing_errors_samples).
    signed error = pred_index - true_index (positive = reconstructed peak late).
    """
    true_peaks = list(np.asarray(true_peaks).astype(int))
    pred_peaks = list(np.asarray(pred_peaks).astype(int))
    used, matches, terr = set(), [], []
    for tp in true_peaks:
        cands = [(abs(pp - tp), pp) for pp in pred_peaks
                 if pp not in used and abs(pp - tp) <= tol_samples]
        if cands:
            _, pp = min(cands)
            used.add(pp); matches.append((tp, pp)); terr.append(pp - tp)
    TP = len(matches)
    FN = len(true_peaks) - TP
    FP = len(pred_peaks) - TP
    return matches, TP, FP, FN, np.asarray(terr, dtype=float)


def temporal_rpeak_metrics(true_ecg, pred_ecg, fs=8000, tol_ms=50,
                           distance=400, prominence=0.5):
    """Full temporal detection report for ONE segment.

    Returns a dict with detection (Se/PPV/F1/FP/FN), timing-error (ms),
    RR-interval error (ms), and heart-rate error (bpm). Segments with no
    ground-truth peaks return n_true=0 and NaN rates (exclude from means, but
    COUNT them -- a reconstruction that yields peaks where truth has none is a
    real failure; see caller handling).
    """
    tol = int(round(tol_ms / 1000 * fs))
    t_pk = detect_r_peaks(true_ecg, fs, distance, prominence)
    p_pk = detect_r_peaks(pred_ecg, fs, distance, prominence)

    matches, TP, FP, FN, terr = match_peaks(t_pk, p_pk, tol)
    sens = TP / (TP + FN) if (TP + FN) else np.nan
    ppv  = TP / (TP + FP) if (TP + FP) else np.nan
    f1   = (2 * sens * ppv / (sens + ppv)
            if (sens and ppv and not np.isnan(sens) and not np.isnan(ppv)
                and (sens + ppv) > 0) else np.nan)

    terr_ms = np.abs(terr) / fs * 1000.0
    mae_t   = float(np.mean(terr_ms)) if terr_ms.size else np.nan
    med_t   = float(np.median(terr_ms)) if terr_ms.size else np.nan
    iqr_t   = float(np.percentile(terr_ms, 75) - np.percentile(terr_ms, 25)) \
              if terr_ms.size else np.nan

    # RR-interval error on consecutive MATCHED pairs only (avoids missed-beat bias)
    rr_err = np.nan
    if len(matches) >= 2:
        tp_sorted = sorted(matches)
        rr_true, rr_pred = [], []
        for (t0, p0), (t1, p1) in zip(tp_sorted[:-1], tp_sorted[1:]):
            rr_true.append((t1 - t0) / fs * 1000.0)
            rr_pred.append((p1 - p0) / fs * 1000.0)
        rr_err = float(np.mean(np.abs(np.array(rr_true) - np.array(rr_pred))))

    # Heart-rate error (bpm), estimator = 60000 / median-RR
    def _hr(pk):
        if len(pk) < 2:
            return np.nan
        return 60000.0 / np.median(np.diff(pk) / fs * 1000.0)
    hr_true, hr_pred = _hr(t_pk), _hr(p_pk)
    hr_err = float(abs(hr_true - hr_pred)) if not (np.isnan(hr_true) or np.isnan(hr_pred)) else np.nan

    return {
        "n_true": int(len(t_pk)), "n_pred": int(len(p_pk)),
        "TP": TP, "FP": FP, "FN": FN,
        "sensitivity": sens, "ppv": ppv, "f1": f1,
        "mae_timing_ms": mae_t, "median_timing_ms": med_t, "iqr_timing_ms": iqr_t,
        "rr_error_ms": rr_err,
        "hr_true_bpm": hr_true, "hr_pred_bpm": hr_pred, "hr_error_bpm": hr_err,
    }


def aggregate_temporal(rows):
    """Aggregate a list of per-segment temporal_rpeak_metrics dicts.

    Detection totals (TP/FP/FN) are POOLED then Se/PPV/F1 recomputed (the
    correct way to aggregate detection over segments). Timing/RR/HR errors are
    averaged over segments where they are defined; the valid-N is reported so
    the denominator is transparent.
    """
    rows = [r for r in rows if r is not None]
    TP = sum(r["TP"] for r in rows)
    FP = sum(r["FP"] for r in rows)
    FN = sum(r["FN"] for r in rows)
    sens = TP / (TP + FN) if (TP + FN) else np.nan
    ppv  = TP / (TP + FP) if (TP + FP) else np.nan
    f1   = 2 * sens * ppv / (sens + ppv) if (sens + ppv) else np.nan

    def _mean(key):
        v = [r[key] for r in rows if r[key] is not None and not np.isnan(r[key])]
        return (float(np.mean(v)), float(np.std(v)), len(v)) if v else (np.nan, np.nan, 0)

    out = {"n_segments": len(rows), "TP": TP, "FP": FP, "FN": FN,
           "sensitivity": sens, "ppv": ppv, "f1": f1}
    for k in ["mae_timing_ms", "median_timing_ms", "iqr_timing_ms",
              "rr_error_ms", "hr_error_bpm"]:
        m, s, n = _mean(k)
        out[k + "_mean"], out[k + "_std"], out[k + "_validN"] = m, s, n
    return out


# ---------------------------------------------------------------------------
# Preprocessing helpers (shared with the paper pipeline)
# ---------------------------------------------------------------------------
def ensure_positive_polarity(ecg):
    """Flip so R-peaks point up. Applied to the ECG TARGET only (never PCG)."""
    ecg = np.asarray(ecg)
    return -ecg if np.abs(ecg.min()) > np.abs(ecg.max()) else ecg


def normalize_zscore(sig):
    sig = np.asarray(sig, dtype=np.float32)
    return ((sig - sig.mean()) / (sig.std() + 1e-8)).astype(np.float32)

In [ ]:
%%writefile /content/ieee_style.py
"""
IEEE double-column figure style for the JBHI revision.

Design goals (per IEEE Transactions / J-BHI figure rules):
  * Vector output (PDF) + high-DPI PNG (600) for every figure.
  * Serif face (Times) to match IEEE body text; DejaVu Serif fallback on Colab.
  * BOLD axis labels that always carry explicit units.
  * Thick spines and ticks so lines stay legible at column scale.
  * constrained_layout everywhere -> no text/label/legend overlaps.
  * Column widths fixed to IEEE spec: single = 3.5 in, double = 7.16 in.

Usage
-----
    from ieee_style import set_ieee_style, new_fig, finalize_axes, save_fig
    set_ieee_style()
    fig, ax = new_fig("single", height=2.2)
    ax.plot(t, y)
    finalize_axes(ax, xlabel="Time (s)", ylabel="Amplitude (a.u.)")
    save_fig(fig, out_dir, "fig_name")   # writes fig_name.pdf AND fig_name.png
"""

import matplotlib
matplotlib.use("Agg")            # headless-safe; remove if you want inline display
import matplotlib.pyplot as plt

# IEEE column widths (inches)
COL_SINGLE = 3.5
COL_DOUBLE = 7.16

# Colour-blind-safe palette (Okabe-Ito), stable role -> colour mapping
COLORS = {
    "truth":    "#009E73",   # bluish green  -> ground truth
    "proposed": "#0072B2",   # blue          -> proposed model
    "cnn":      "#D55E00",   # vermillion    -> CNN
    "unet":     "#CC79A7",   # reddish purple-> U-Net
    "wgan":     "#E69F00",   # orange        -> WGAN-GP
    "pcg":      "#666666",   # grey          -> PCG input
    "accent":   "#0072B2",
    "muted":    "#999999",
}


def set_ieee_style():
    """Apply the global rcParams. Call once at the top of every script."""
    plt.rcParams.update({
        "figure.dpi":        150,
        "savefig.dpi":       600,
        "savefig.bbox":      "tight",
        "savefig.pad_inches": 0.02,
        "figure.constrained_layout.use": True,
        "font.size":         8,
        "font.family":       "serif",
        "font.serif":        ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset":  "dejavuserif",
        "text.usetex":       False,
        "axes.labelsize":    9,
        "axes.labelweight":  "bold",
        "axes.titlesize":    9,
        "axes.titleweight":  "bold",
        "axes.linewidth":    1.1,
        "axes.grid":         True,
        "grid.alpha":        0.25,
        "grid.linewidth":    0.4,
        "xtick.labelsize":   8,
        "ytick.labelsize":   8,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.major.size":  3.5,
        "ytick.major.size":  3.5,
        "xtick.direction":   "out",
        "ytick.direction":   "out",
        "legend.fontsize":   7.5,
        "legend.framealpha": 0.92,
        "legend.edgecolor":  "0.7",
        "legend.handlelength": 1.6,
        "lines.linewidth":   1.0,
        "lines.solid_capstyle": "round",
    })


def new_fig(width="single", height=2.2, nrows=1, ncols=1, **kw):
    """Create a figure sized to an IEEE column.

    width : 'single' (3.5 in), 'double' (7.16 in), or a float in inches.
    height: figure height in inches.
    Returns (fig, ax) or (fig, axes) exactly like plt.subplots.
    """
    w = {"single": COL_SINGLE, "double": COL_DOUBLE}.get(width, width)
    fig, ax = plt.subplots(nrows, ncols, figsize=(w, height), **kw)
    return fig, ax


def finalize_axes(ax, xlabel=None, ylabel=None, title=None,
                  legend=False, legend_loc="best", legend_ncol=1):
    """Apply bold labels with units, thicken spines, tidy ticks.

    Always pass xlabel/ylabel WITH units, e.g. 'Time (ms)', 'PSNR (dB)'.
    """
    if xlabel:
        ax.set_xlabel(xlabel, fontweight="bold")
    if ylabel:
        ax.set_ylabel(ylabel, fontweight="bold")
    if title:
        ax.set_title(title, fontweight="bold")
    for spine in ax.spines.values():
        spine.set_linewidth(1.1)
    ax.tick_params(axis="both", which="major", width=1.0, labelsize=8)
    if legend:
        leg = ax.legend(loc=legend_loc, ncol=legend_ncol, frameon=True)
        leg.get_frame().set_linewidth(0.6)
    return ax


def save_fig(fig, out_directory, name):
    """Save both a vector PDF (for LaTeX) and a 600-dpi PNG (for preview)."""
    import os
    pdf = os.path.join(out_directory, f"{name}.pdf")
    png = os.path.join(out_directory, f"{name}.png")
    fig.savefig(pdf)
    fig.savefig(png)
    plt.close(fig)
    print(f"  saved  {pdf}")
    print(f"  saved  {png}")
    return pdf, png

In [ ]:
import importlib
import config_paths, models, metrics, ieee_style
for m in (config_paths, models, metrics, ieee_style):
    importlib.reload(m)
ieee_style.set_ieee_style()
print('shared modules ready | device =', config_paths.DEVICE)

## Experiment

# exp6 - Component Ablation Training Harness (Reviewer 1, comment 4)

Reviewer 1 asked for a controlled ablation that isolates each design choice in
the proposed generator. This script trains every ablation variant under ONE
fixed protocol, then scores each trained model against a SINGLE
polarity-corrected reference so the numbers are directly comparable.

The grouped factors under test are:
  * MHSA          (multi-head self-attention in the 4 bottleneck blocks)
  * MSDC          (multi-scale dilated convolutions in the bottleneck)
  * output head   (batchnorm | tanh | linear)
  * adversarial   (conditional GAN loss on/off; L1-only when off)
  * polarity      (data-loader polarity correction on/off)

Design notes
------------
* MHSA and MSDC are ARCHITECTURE switches on models.ConfigurableGenerator.
* The output head is an ARCHITECTURE switch (output_activation).
* Adversarial on/off is a TRAINING switch (drop the GAN term, keep L1).
* Polarity on/off is a DATA-LOADER switch (skip metrics.ensure_positive_polarity).

The harness is RESUMABLE. Each variant writes a per-epoch resume checkpoint and
a final checkpoint. Re-running skips any variant whose final checkpoint exists,
and resumes any variant that has a partial resume checkpoint.

IMPORTANT: this is the heaviest job in the revision. It is meant to run on a
Colab GPU over several sessions. Nothing runs on the author's laptop.

## Dependencies
scikit-image supplies the SSIM used by metrics.calc_ssim. The rest ship with
Colab. Uncomment the pip line below on Colab if scikit-image is missing.

## Setup (mount Drive, put the shared modules on the path)

Copy your local `Revision/Colab_Experiments/shared/` folder to
`MyDrive/JBHI_Revision/shared/` on Drive, OR paste the four shared modules into
`/content`. The self-contained notebook built by `build_notebooks.py` writes the
modules to `/content` for you, so it does not need this cell.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Not on Colab (or Drive already mounted):', exc)

import sys
SHARED_ON_DRIVE = '/content/drive/MyDrive/JBHI_Revision/shared'
if SHARED_ON_DRIVE not in sys.path:
    sys.path.append(SHARED_ON_DRIVE)
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

print('NOTE: this script imports config_paths, models, metrics, ieee_style.')
print('      Copy the local shared/ folder to', SHARED_ON_DRIVE)
print('      or paste the four shared modules into /content before running.')

import config_paths
import models
import metrics
import ieee_style

ieee_style.set_ieee_style()
print('shared modules ready | device =', config_paths.DEVICE)

In [ ]:
import os
import glob
import json
import time
import copy
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = config_paths.DEVICE
OUT = config_paths.out_dir('exp6_ablation')          # all artifacts land here
CKPT_DIR = os.path.join(OUT, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)
print('output dir     :', OUT)
print('checkpoint dir :', CKPT_DIR)

## Fixed training protocol (identical for every variant)

100 epochs, Adam(5e-5, betas 0.5/0.999), cosine schedule to 1e-6, gradient
clipping at 1.0, batch size 8, L1 weight 100, He initialization on the
generator. The subject split is read from `config_paths.SPLIT_FILE`, so every
variant trains and tests on exactly the same subjects.

In [ ]:
EPOCHS       = 60      # 60 is ample for the ablation: component deltas converge
                       # well before 100, and this cuts wall-clock ~40%.
LR           = 5e-5
BETAS        = (0.5, 0.999)
ETA_MIN      = 1e-6
GRAD_CLIP    = 1.0
LAMBDA_L1    = 100.0
SEED         = 42

# ---- Hardware-aware knobs (auto-scale to the GPU; override by hand if needed) --
# The free T4 (14.5 GB) OOM-ed the attention variants at batch 8. Two fixes make
# an A100 fast and OOM-free: (1) models.MultiHeadSelfAttention now uses
# memory-efficient attention, and (2) bf16 autocast halves activation memory and
# roughly doubles throughput on A100 tensor cores. Spend the compute-unit budget
# on a large batch: an A100-80GB runs batch 64 comfortably, an A100-40GB batch 32.
torch.backends.cudnn.benchmark = True    # fixed 64000-sample inputs -> autotune
torch.backends.cuda.matmul.allow_tf32 = True     # TF32 matmul on Ampere (A100)
torch.backends.cudnn.allow_tf32 = True
_cuda = torch.cuda.is_available()
_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if _cuda else 0.0
# SDPA (flash / memory-efficient attention in models.MultiHeadSelfAttention) removes
# the O(T^2) attention-memory wall, so push the batch up to use the A100 fully.
#   A100-80GB -> 128, A100-40GB -> 64, mid GPUs -> 16, T4/CPU -> 8. Flash attention
#   makes the attention variants fit at these sizes; a bigger batch means fewer
#   optimizer steps per epoch, which is the main throughput lever on the A100.
BATCH_SIZE  = (128 if _gb > 60 else 64 if _gb > 34 else 16 if _gb > 20 else 8) if _cuda else 8
NUM_WORKERS = min(8, (os.cpu_count() or 4)) if _cuda else 2
USE_AMP     = _cuda and torch.cuda.is_bf16_supported()     # A100 supports bf16
AMP_DTYPE   = torch.bfloat16
USE_COMPILE = _cuda   # torch.compile the generator (~1.3-1.8x on A100); set False
                      # if you hit a compile error. The critic stays eager because
                      # spectral norm + torch.compile can misbehave.
PRELOAD_TO_RAM = True     # hold the whole train set in RAM to remove per-epoch Drive
                          # I/O (the dominant Colab cost; it drove epochs to ~700 s on T4)
print(f'GPU: {_gb:.0f} GB | bf16 AMP: {USE_AMP} | batch: {BATCH_SIZE} | workers: {NUM_WORKERS}')

# Evaluation tolerances for the temporal R-peak F1 (samples <- ms via FS).
TEMPORAL_TOL_MS = [50, 150]

# R1.8 CONFOUND GUARD: the paper's inference pipeline rescales the generator
# output by the target standard deviation (fake *= real_std / fake_std). That
# rescaling leaks the ground-truth scale into the prediction and is exactly the
# confound Reviewer 1 comment 8 exposes. This ablation reports BLIND numbers, so
# NO test-time rescaling is applied. Keep this False.
EVAL_RESCALE_BY_TARGET_STD = False


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## Ablation variants

Each entry is one row of the ablation table. `use_attention` and `use_dilated`
toggle MHSA and MSDC. `output` selects the output head. `adversarial` toggles
the GAN term. `polarity` toggles data-loader polarity correction.

VERIFIED generator parameter counts (computed locally with torch; the main
text's "14.0M" is an error, use these):
  U-Net only              10.92M   (att F, dil F)
  U-Net + MSDC            15.12M   (att F, dil T)
  U-Net + MHSA            15.11M   (att T, dil F)
  U-Net + both (proposed) 19.32M   (att T, dil T)  = 19,315,715 params
  Spectral-norm critic     2.18M   (2,176,833 params)
The tanh and linear heads share the +both backbone. They carry no output-head
affine parameters, so they hold 2 fewer parameters than the BatchNorm head
(19,315,713 vs 19,315,715); both round to 19.32M. The adversarial-off and
polarity-off variants use the +both architecture unchanged (19.32M); they only
change the loss or the data loader. The harness prints count_params at build
time so you can confirm every number against these comments.

In [ ]:
VARIANTS = {
    # key                 attention  dilated  output        adversarial polarity  params(M)
    'unet_only':        dict(use_attention=False, use_dilated=False, output='batchnorm', adversarial=True,  polarity=True,  params_m=10.92),
    'msdc':             dict(use_attention=False, use_dilated=True,  output='batchnorm', adversarial=True,  polarity=True,  params_m=15.12),
    'mhsa':             dict(use_attention=True,  use_dilated=False, output='batchnorm', adversarial=True,  polarity=True,  params_m=15.11),
    'both':             dict(use_attention=True,  use_dilated=True,  output='batchnorm', adversarial=True,  polarity=True,  params_m=19.32),
    'proposed_tanh':    dict(use_attention=True,  use_dilated=True,  output='tanh',      adversarial=True,  polarity=True,  params_m=19.32),
    'proposed_linear':  dict(use_attention=True,  use_dilated=True,  output='linear',    adversarial=True,  polarity=True,  params_m=19.32),
    # 'proposed_batchnorm' is IDENTICAL to 'both' (proposed model, batchnorm head,
    # adversarial, polarity-corrected). Train one, then reuse its checkpoint for
    # the other row if you want the label in the table.
    'proposed_batchnorm': dict(use_attention=True, use_dilated=True, output='batchnorm', adversarial=True,  polarity=True,  params_m=19.32),
    # Training-protocol / data variants of the proposed model (architecture 19.32M):
    'adv_off':          dict(use_attention=True,  use_dilated=True,  output='batchnorm', adversarial=False, polarity=True,  params_m=19.32),
    'polarity_off':     dict(use_attention=True,  use_dilated=True,  output='batchnorm', adversarial=True,  polarity=False, params_m=19.32),
}

# ---------------------------------------------------------------------------
# TRIAGE ORDER (run these first; each answers a distinct reviewer worry):
#   1. polarity_off @ 100 epochs  -> shows polarity correction is load-bearing
#                                     (its blind correlation should collapse).
#   2. proposed_tanh              -> shows the output head choice matters.
#   3. mhsa                       -> isolates attention.
#   4. msdc                       -> isolates dilated multi-scale context.
#   5. unet_only                  -> the plain backbone floor.
# 'both' reproduces the proposed model already in the paper (PSNR 17.10 dB,
# corr 0.610), so retrain it only if you want a same-protocol control.
# ---------------------------------------------------------------------------
# Full set for a complete R1.4 table. Variants whose final checkpoint already
# exists are auto-skipped, so re-running only trains what is missing. The
# attention variants (mhsa, both, proposed_tanh/linear, polarity_off, adv_off)
# are the ones that OOM-ed before; the flash-attention + batch-32 fix lets them
# fit on a 40 GB A100. (msdc, unet_only already completed at batch 48 -- delete
# their checkpoints to re-run at the uniform batch 32 if you want one protocol.)
RUN_VARIANTS = ['mhsa', 'polarity_off', 'proposed_tanh', 'proposed_linear',
                'adv_off', 'both', 'msdc', 'unet_only']

## Verify the parameter counts before spending GPU hours

This instantiates each variant's generator and the critic, then prints
count_params next to the verified comment values. Confirm they match before you
launch a 100-epoch run.

In [ ]:
def build_generator(cfg):
    """Construct a ConfigurableGenerator for a variant config dict."""
    return models.ConfigurableGenerator(
        base_channels=64,
        num_scenarios=config_paths.NUM_SCENARIOS,
        use_attention=cfg['use_attention'],
        use_dilated=cfg['use_dilated'],
        output_activation=cfg['output'],
        use_scenario=True,
    )


def build_discriminator():
    return models.SpectralNormDiscriminator(
        base_channels=64, num_scenarios=config_paths.NUM_SCENARIOS)


print(f'{"variant":<20} {"gen params":>12} {"expected(M)":>12}')
print('-' * 46)
for _name, _cfg in VARIANTS.items():
    _g = build_generator(_cfg)
    _p = models.count_params(_g)
    print(f'{_name:<20} {_p:>12,} {_cfg["params_m"]:>11.2f}M')
    del _g
_d = build_discriminator()
print(f'{"discriminator":<20} {models.count_params(_d):>12,} {"2.18":>11}M')
del _d

## Dataset (EPHNOGRAM chunks with a polarity switch)

One item per `.npz` chunk. The ECG target is polarity-corrected only when the
variant asks for it. Both signals are z-scored, matching the paper pipeline.
The scenario one-hot conditions the generator and the critic.

In [ ]:
def _load_split():
    """Return train/val/test subject sets from config_paths.SPLIT_FILE."""
    with open(config_paths.SPLIT_FILE) as f:
        sp = json.load(f)

    def pick(*keys):
        for k in keys:
            if k in sp:
                return set(str(s) for s in sp[k])
        return set()

    train = pick('train_subjects', 'train')
    val = pick('val_subjects', 'val')
    test = pick('test_subjects', 'test')
    return train, val, test


def _index_chunks():
    """Map subject_id -> [chunk paths]. Cached to JSON so we scan Drive once.

    np.load on an .npz is lazy, so reading only the 'subject_id' key does not
    decompress the full ecg/pcg arrays.
    """
    cache = os.path.join(OUT, 'chunk_index.json')
    if os.path.exists(cache):
        with open(cache) as f:
            return {k: v for k, v in json.load(f).items()}
    index = {}
    paths = sorted(glob.glob(os.path.join(config_paths.EPH_CHUNKS_DIR, '*.npz')))
    for p in paths:
        try:
            sid = str(np.load(p, allow_pickle=True)['subject_id'])
        except Exception as exc:
            print('  skip (unreadable):', os.path.basename(p), exc)
            continue
        index.setdefault(sid, []).append(p)
    with open(cache, 'w') as f:
        json.dump(index, f, indent=1)
    return index


class EphnogramChunks(Dataset):
    """PCG -> ECG paired chunks for a set of subjects.

    polarity=True  applies metrics.ensure_positive_polarity to the raw ECG before
                   z-scoring (the paper pipeline).
    polarity=False leaves the raw ECG polarity untouched, so a model can learn an
                   inverted target. This is the polarity_off ablation.
    """
    def __init__(self, chunk_paths, polarity=True):
        self.paths = list(chunk_paths)
        self.polarity = polarity

    def __len__(self):
        return len(self.paths)

    def _fit_length(self, arr):
        arr = np.asarray(arr, dtype=np.float32).ravel()
        n = config_paths.SEG_LEN
        if arr.shape[0] >= n:
            return arr[:n]
        pad = np.zeros(n - arr.shape[0], dtype=np.float32)
        return np.concatenate([arr, pad])

    def __getitem__(self, idx):
        d = np.load(self.paths[idx], allow_pickle=True)
        ecg_raw = self._fit_length(d['ecg'])
        pcg_raw = self._fit_length(d['pcg'])

        if self.polarity:
            ecg_raw = metrics.ensure_positive_polarity(ecg_raw)
        ecg = metrics.normalize_zscore(ecg_raw)
        pcg = metrics.normalize_zscore(pcg_raw)

        sc_idx = config_paths.scenario_to_index(d['scenario'])
        scen = np.zeros(config_paths.NUM_SCENARIOS, dtype=np.float32)
        scen[sc_idx] = 1.0

        pcg_t = torch.from_numpy(pcg).unsqueeze(0)   # (1, L)
        ecg_t = torch.from_numpy(ecg).unsqueeze(0)   # (1, L)
        scen_t = torch.from_numpy(scen)              # (3,)
        return pcg_t, ecg_t, scen_t


# ---------------------------------------------------------------------------
# In-RAM training set. Reading 4,059 .npz from Drive every epoch is the dominant
# Colab cost (it drove epoch time to ~700 s on the T4). The whole train set is
# only ~2 GB of float32, so with 84 GB RAM we load the RAW arrays ONCE and reuse
# them across every variant. Workers fork this array copy-on-write, so no copies.
# ---------------------------------------------------------------------------
_RAM_CACHE = {}   # tag -> (pcg_all, ecg_all, scen_all) raw, shared across variants


def _fit_len(arr):
    arr = np.asarray(arr, dtype=np.float32).ravel()
    n = config_paths.SEG_LEN
    if arr.shape[0] >= n:
        return arr[:n]
    return np.concatenate([arr, np.zeros(n - arr.shape[0], dtype=np.float32)])


def _preload_raw(paths, tag):
    if tag in _RAM_CACHE:
        return _RAM_CACHE[tag]
    n, L = len(paths), config_paths.SEG_LEN
    pcg = np.zeros((n, L), dtype=np.float32)
    ecg = np.zeros((n, L), dtype=np.float32)
    scen = np.zeros(n, dtype=np.int64)
    t0 = time.time()
    for i, p in enumerate(paths):
        d = np.load(p, allow_pickle=True)
        pcg[i] = _fit_len(d['pcg'])
        ecg[i] = _fit_len(d['ecg'])
        scen[i] = config_paths.scenario_to_index(d['scenario'])
        if (i + 1) % 500 == 0:
            print(f'    preloaded {i + 1}/{n} chunks into RAM')
    print(f'  RAM preload [{tag}]: {n} chunks, '
          f'{(pcg.nbytes + ecg.nbytes) / 1e9:.2f} GB, {time.time() - t0:.0f}s')
    _RAM_CACHE[tag] = (pcg, ecg, scen)
    return _RAM_CACHE[tag]


class InMemoryChunks(Dataset):
    """Whole subject set held in RAM. Polarity + z-score run per item (cheap)."""
    def __init__(self, paths, tag, polarity=True):
        self.pcg, self.ecg, self.scen = _preload_raw(paths, tag)
        self.polarity = polarity

    def __len__(self):
        return self.pcg.shape[0]

    def __getitem__(self, idx):
        ecg_raw = self.ecg[idx]
        if self.polarity:
            ecg_raw = metrics.ensure_positive_polarity(ecg_raw)
        ecg = metrics.normalize_zscore(ecg_raw)
        pcg = metrics.normalize_zscore(self.pcg[idx])
        scen = np.zeros(config_paths.NUM_SCENARIOS, dtype=np.float32)
        scen[int(self.scen[idx])] = 1.0
        return (torch.from_numpy(pcg).unsqueeze(0),
                torch.from_numpy(ecg).unsqueeze(0),
                torch.from_numpy(scen))


def make_loaders(polarity):
    train_subj, _val_subj, test_subj = _load_split()
    index = _index_chunks()
    train_paths = [p for s in sorted(train_subj) for p in index.get(s, [])]
    test_paths = [p for s in sorted(test_subj) for p in index.get(s, [])]
    print(f'  train subjects {sorted(train_subj)} -> {len(train_paths)} chunks')
    print(f'  test  subjects {sorted(test_subj)} -> {len(test_paths)} chunks')
    if PRELOAD_TO_RAM:
        train_ds = InMemoryChunks(train_paths, tag='train', polarity=polarity)
    else:
        train_ds = EphnogramChunks(train_paths, polarity=polarity)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, drop_last=True,
                              persistent_workers=(NUM_WORKERS > 0),
                              pin_memory=(DEVICE.type == 'cuda'),
                              prefetch_factor=(4 if NUM_WORKERS > 0 else None))
    # The test loader always uses polarity-corrected references so every variant
    # is scored against the SAME ground truth (see evaluate_variant).
    return train_loader, test_paths

## He initialization (generator only)

The spectral-norm critic keeps its default initialization. He (Kaiming) init on
a spectrally-normalized weight would not persist, because that weight is a
derived buffer rather than a leaf parameter.

In [ ]:
def he_init(module):
    for m in module.modules():
        if isinstance(m, (nn.Conv1d, nn.ConvTranspose1d, nn.Linear)):
            nn.init.kaiming_normal_(m.weight, a=0, nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.BatchNorm1d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

## Trainer (matched protocol, resumable)

The generator loss is the L1 term (weight 100) plus, when adversarial is on, a
conditional-GAN BCE term on the spectral-norm critic. The critic sees the PCG
condition concatenated with a real or generated ECG. A per-epoch resume file
lets an interrupted run continue; a final checkpoint marks the variant done.

In [ ]:
def _paths_for(name):
    final_ckpt = os.path.join(CKPT_DIR, f'{name}_final.pt')
    resume_ckpt = os.path.join(CKPT_DIR, f'{name}_resume.pt')
    return final_ckpt, resume_ckpt


def _maybe_compile(module):
    """Return a torch.compile-d module for a fast forward, or the module itself.

    The compiled wrapper SHARES parameters with `module`, so we always save/load
    the underlying `module.state_dict()` (no '_orig_mod.' prefix) and only use the
    compiled object for forward passes.
    """
    if not USE_COMPILE:
        return module
    try:
        return torch.compile(module)
    except Exception as exc:
        print('  torch.compile unavailable -> eager:', exc)
        return module


def train_variant(name, cfg):
    """Train one variant under the fixed protocol. Returns the final ckpt path.

    Skips entirely if the final checkpoint exists. Resumes from the per-epoch
    resume file when present.
    """
    final_ckpt, resume_ckpt = _paths_for(name)
    if os.path.exists(final_ckpt):
        print(f'[{name}] final checkpoint present -> skip training.')
        return final_ckpt

    set_seed(SEED)
    adversarial = cfg['adversarial']
    train_loader, _test_paths = make_loaders(polarity=cfg['polarity'])

    gen = build_generator(cfg).to(DEVICE)
    he_init(gen)
    gen_fwd = _maybe_compile(gen)     # compiled forward; `gen` stays the leaf for state_dict
    opt_g = torch.optim.Adam(gen.parameters(), lr=LR, betas=BETAS)
    sched_g = torch.optim.lr_scheduler.CosineAnnealingLR(opt_g, T_max=EPOCHS, eta_min=ETA_MIN)

    disc = opt_d = sched_d = None
    if adversarial:
        disc = build_discriminator().to(DEVICE)
        opt_d = torch.optim.Adam(disc.parameters(), lr=LR, betas=BETAS)
        sched_d = torch.optim.lr_scheduler.CosineAnnealingLR(opt_d, T_max=EPOCHS, eta_min=ETA_MIN)

    l1 = nn.L1Loss()
    bce = nn.BCELoss()   # the critic already applies a sigmoid

    start_epoch = 0
    history = {'epoch': [], 'loss_g': [], 'loss_d': [], 'loss_l1': []}
    if os.path.exists(resume_ckpt):
        st = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
        gen.load_state_dict(st['gen'])
        opt_g.load_state_dict(st['opt_g'])
        sched_g.load_state_dict(st['sched_g'])
        if adversarial and st.get('disc') is not None:
            disc.load_state_dict(st['disc'])
            opt_d.load_state_dict(st['opt_d'])
            sched_d.load_state_dict(st['sched_d'])
        history = st['history']
        start_epoch = st['epoch'] + 1
        print(f'[{name}] resuming at epoch {start_epoch}/{EPOCHS}.')

    for ep in range(start_epoch, EPOCHS):
        gen.train()
        if adversarial:
            disc.train()
        t0 = time.time()
        run_g = run_d = run_l1 = 0.0
        n_batches = 0
        for pcg, ecg_real, scen in train_loader:
            pcg = pcg.to(DEVICE, non_blocking=True)
            ecg_real = ecg_real.to(DEVICE, non_blocking=True)
            scen = scen.to(DEVICE, non_blocking=True)

            ac = torch.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP)
            # ---- discriminator step ----
            if adversarial:
                with torch.no_grad(), ac:
                    fake_detached = gen_fwd(pcg, scen)
                with ac:
                    d_real = disc(pcg, ecg_real, scen)
                    d_fake = disc(pcg, fake_detached, scen)
                # BCE is autocast-UNSAFE (raises under autocast) -> compute in fp32.
                loss_d = bce(d_real.float(), torch.ones_like(d_real, dtype=torch.float32)) + \
                         bce(d_fake.float(), torch.zeros_like(d_fake, dtype=torch.float32))
                opt_d.zero_grad(set_to_none=True)
                loss_d.backward()
                nn.utils.clip_grad_norm_(disc.parameters(), GRAD_CLIP)
                opt_d.step()
                run_d += float(loss_d.detach())

            # ---- generator step ----
            with ac:
                fake = gen_fwd(pcg, scen)
                loss_l1 = l1(fake, ecg_real)
                d_fake_for_g = disc(pcg, fake, scen) if adversarial else None
            if adversarial:                          # BCE in fp32, outside autocast
                loss_g = bce(d_fake_for_g.float(),
                             torch.ones_like(d_fake_for_g, dtype=torch.float32)) \
                         + LAMBDA_L1 * loss_l1.float()
            else:
                loss_g = LAMBDA_L1 * loss_l1.float()   # L1-only, matched L1 weighting
            opt_g.zero_grad(set_to_none=True)
            loss_g.backward()
            nn.utils.clip_grad_norm_(gen.parameters(), GRAD_CLIP)
            opt_g.step()

            run_g += float(loss_g.detach())
            run_l1 += float(loss_l1.detach())
            n_batches += 1

        sched_g.step()
        if adversarial:
            sched_d.step()

        n_batches = max(n_batches, 1)
        history['epoch'].append(ep)
        history['loss_g'].append(run_g / n_batches)
        history['loss_d'].append(run_d / n_batches)
        history['loss_l1'].append(run_l1 / n_batches)

        torch.save({
            'epoch': ep, 'variant': name, 'config': cfg,
            'gen': gen.state_dict(),
            'disc': disc.state_dict() if adversarial else None,
            'opt_g': opt_g.state_dict(),
            'opt_d': opt_d.state_dict() if adversarial else None,
            'sched_g': sched_g.state_dict(),
            'sched_d': sched_d.state_dict() if adversarial else None,
            'history': history,
        }, resume_ckpt)

        if ep % 5 == 0 or ep == EPOCHS - 1:
            print(f'[{name}] epoch {ep:3d}/{EPOCHS}  '
                  f'G {history["loss_g"][-1]:.4f}  '
                  f'D {history["loss_d"][-1]:.4f}  '
                  f'L1 {history["loss_l1"][-1]:.4f}  '
                  f'({time.time() - t0:.1f}s)')

    torch.save({
        'variant': name, 'config': cfg, 'epochs': EPOCHS,
        'gen': gen.state_dict(), 'history': history,
        'params': models.count_params(gen),
    }, final_ckpt)
    print(f'[{name}] done -> {final_ckpt}')
    return final_ckpt

## Evaluation (same polarity-corrected reference for every variant)

Each trained generator is scored on the held-out test subjects. The reference
ECG is always polarity-corrected and z-scored, so every variant, including
polarity_off, is judged against identical ground truth. No target-std rescaling
is applied (see the R1.8 confound guard). Signal fidelity uses metrics.calc_*;
the temporal F1 uses metrics.temporal_rpeak_metrics aggregated over segments.

In [ ]:
def _scenario_onehot(scenario_value):
    scen = np.zeros(config_paths.NUM_SCENARIOS, dtype=np.float32)
    scen[config_paths.scenario_to_index(scenario_value)] = 1.0
    return scen


def evaluate_variant(name, cfg):
    """Return a results dict of blind test metrics for a trained variant."""
    final_ckpt, _ = _paths_for(name)
    if not os.path.exists(final_ckpt):
        print(f'[{name}] no final checkpoint -> cannot evaluate.')
        return None

    st = torch.load(final_ckpt, map_location=DEVICE, weights_only=False)
    gen = build_generator(cfg).to(DEVICE)
    gen.load_state_dict(st['gen'])
    gen.eval()

    _train, _val, test_subj = _load_split()
    index = _index_chunks()
    test_paths = [p for s in sorted(test_subj) for p in index.get(s, [])]

    psnr_v, ssim_v, corr_v, ramp_v = [], [], [], []
    temporal_rows = {tol: [] for tol in TEMPORAL_TOL_MS}

    with torch.no_grad():
        for p in test_paths:
            d = np.load(p, allow_pickle=True)
            ecg_raw = np.asarray(d['ecg'], dtype=np.float32).ravel()[:config_paths.SEG_LEN]
            pcg_raw = np.asarray(d['pcg'], dtype=np.float32).ravel()[:config_paths.SEG_LEN]
            if ecg_raw.shape[0] < config_paths.SEG_LEN:
                continue

            # FIXED reference: always polarity-corrected, always z-scored.
            ref = metrics.normalize_zscore(metrics.ensure_positive_polarity(ecg_raw))
            pcg = metrics.normalize_zscore(pcg_raw)

            scen = torch.from_numpy(_scenario_onehot(d['scenario'])).unsqueeze(0).to(DEVICE)
            pcg_t = torch.from_numpy(pcg).unsqueeze(0).unsqueeze(0).to(DEVICE)
            fake = gen(pcg_t, scen).squeeze().cpu().numpy()

            if EVAL_RESCALE_BY_TARGET_STD:      # kept False; see R1.8 guard
                fake = fake * (ref.std() / (fake.std() + 1e-8))

            psnr_v.append(metrics.calc_psnr(ref, fake))
            corr_v.append(metrics.calc_correlation(ref, fake))
            try:
                ssim_v.append(metrics.calc_ssim(ref, fake))
            except Exception:
                pass
            ramp = metrics.calc_r_peak_amplitude_error(
                ref, fake, fs=config_paths.FS,
                distance=config_paths.RPEAK_DISTANCE,
                prominence=config_paths.RPEAK_PROMINENCE)
            if not np.isnan(ramp):
                ramp_v.append(ramp)

            for tol in TEMPORAL_TOL_MS:
                temporal_rows[tol].append(metrics.temporal_rpeak_metrics(
                    ref, fake, fs=config_paths.FS, tol_ms=tol,
                    distance=config_paths.RPEAK_DISTANCE,
                    prominence=config_paths.RPEAK_PROMINENCE))

    def _m(vals):
        return float(np.mean(vals)) if len(vals) else float('nan')

    result = {
        'variant': name,
        'config': cfg,
        'params': int(st.get('params', models.count_params(gen))),
        'n_test_segments': len(psnr_v),
        'psnr': _m(psnr_v),
        'ssim': _m(ssim_v),
        'corr': _m(corr_v),
        'rpeak_amp_err': _m(ramp_v),
        'rpeak_amp_err_validN': len(ramp_v),
    }
    for tol in TEMPORAL_TOL_MS:
        agg = metrics.aggregate_temporal(temporal_rows[tol])
        result[f'temporal_f1_{tol}'] = agg['f1']
        result[f'temporal_se_{tol}'] = agg['sensitivity']
        result[f'temporal_ppv_{tol}'] = agg['ppv']

    with open(os.path.join(OUT, f'metrics_{name}.json'), 'w') as f:
        json.dump(result, f, indent=2)
    print(f'[{name}] PSNR {result["psnr"]:.2f} dB | corr {result["corr"]:.3f} | '
          f'F1@50 {result.get("temporal_f1_50", float("nan")):.3f} | '
          f'params {result["params"]:,}')
    return result

## Run the queued variants

Trains each variant in RUN_VARIANTS (skipping finished ones), evaluates it, and
accumulates results into `all_results.json`. Every step is guarded, so a missing
data path prints a message and the loop continues.

In [ ]:
def load_all_results():
    path = os.path.join(OUT, 'all_results.json')
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return {}


def save_all_results(all_results):
    with open(os.path.join(OUT, 'all_results.json'), 'w') as f:
        json.dump(all_results, f, indent=2)


all_results = load_all_results()
for _name in RUN_VARIANTS:
    if _name not in VARIANTS:
        print(f'[{_name}] not a known variant -> skip.')
        continue
    _cfg = VARIANTS[_name]
    print('=' * 64)
    print(f'VARIANT: {_name}  {_cfg}')
    print('=' * 64)
    try:
        train_variant(_name, _cfg)
        res = evaluate_variant(_name, _cfg)
        if res is not None:
            all_results[_name] = res
            save_all_results(all_results)
    except FileNotFoundError as exc:
        print(f'[{_name}] data or checkpoint path missing -> skipped:', exc)
    except Exception as exc:
        print(f'[{_name}] error -> skipped:', exc)
    finally:
        # release this variant's GPU memory before the next one starts, so a
        # failed variant cannot starve the following variants.
        import gc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print('\nfinished variants:', sorted(all_results.keys()))

## Summary figure (IEEE double-column)

Grouped bars of PSNR, Pearson correlation, and temporal F1 at plus/minus 50 ms
across the evaluated variants. Bold axis labels carry explicit units, and
constrained_layout keeps the rotated variant labels clear of the bars.

In [ ]:
def plot_summary(all_results, tol=50):
    results = load_all_results() if not all_results else all_results
    order = [k for k in VARIANTS if k in results]
    if not order:
        print('No results yet -> skipping summary figure.')
        return

    labels = order
    psnr = [results[k]['psnr'] for k in order]
    corr = [results[k]['corr'] for k in order]
    f1 = [results[k].get(f'temporal_f1_{tol}', float('nan')) for k in order]

    palette = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00',
               '#56B4E9', '#666666', '#F0E442', '#000000']
    colors = [palette[i % len(palette)] for i in range(len(order))]

    fig, axes = ieee_style.new_fig('double', height=2.7, ncols=3)
    panels = [
        (axes[0], psnr, 'PSNR (dB)'),
        (axes[1], corr, 'Pearson r (a.u.)'),
        (axes[2], f1, f'Temporal F1 at {tol} ms (a.u.)'),
    ]
    x = np.arange(len(order))
    for ax, vals, ylab in panels:
        ax.bar(x, vals, color=colors, width=0.72, edgecolor='black', linewidth=0.5)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=6.5)
        ieee_style.finalize_axes(ax, ylabel=ylab)
    ieee_style.save_fig(fig, OUT, 'fig_exp6_ablation_summary')


def plot_training_curves(name):
    """Optional per-variant G/D/L1 loss curves (IEEE single-column)."""
    final_ckpt, _ = _paths_for(name)
    if not os.path.exists(final_ckpt):
        print(f'[{name}] no checkpoint -> skip curves.')
        return
    hist = torch.load(final_ckpt, map_location='cpu', weights_only=False)['history']
    if not hist['epoch']:
        return
    fig, ax = ieee_style.new_fig('single', height=2.3)
    ax.plot(hist['epoch'], hist['loss_l1'], color=ieee_style.COLORS['proposed'], label='L1')
    ax.plot(hist['epoch'], hist['loss_g'], color=ieee_style.COLORS['cnn'], label='Generator')
    if any(v > 0 for v in hist['loss_d']):
        ax.plot(hist['epoch'], hist['loss_d'], color=ieee_style.COLORS['unet'], label='Discriminator')
    ieee_style.finalize_axes(ax, xlabel='Epoch (count)', ylabel='Loss (a.u.)', legend=True)
    ieee_style.save_fig(fig, OUT, f'fig_exp6_curves_{name}')


plot_summary(all_results, tol=50)
for _name in RUN_VARIANTS:
    if _name in VARIANTS:
        plot_training_curves(_name)

## LaTeX ablation table skeleton

Writes a paste-ready table to `table_exp6_ablation.tex` in the output folder.
Cells fill from `all_results.json` where a variant has been evaluated, otherwise
they hold a placeholder dash. Copy the file into the manuscript and drop it near
the ablation discussion.

In [ ]:
def emit_latex_table(all_results):
    results = load_all_results() if not all_results else all_results

    display = {
        'unet_only':          'U-Net only',
        'msdc':               'U-Net + MSDC',
        'mhsa':               'U-Net + MHSA',
        'both':               'U-Net + both',
        'proposed_batchnorm': 'Proposed (BatchNorm)',
        'proposed_tanh':      'Proposed + Tanh',
        'proposed_linear':    'Proposed + linear',
        'adv_off':            'Proposed, adversarial off',
        'polarity_off':       'Proposed, polarity off',
    }
    row_order = ['unet_only', 'msdc', 'mhsa', 'both', 'proposed_batchnorm',
                 'proposed_tanh', 'proposed_linear', 'adv_off', 'polarity_off']

    yes = r'\checkmark'          # amssymb provides \checkmark (IEEEtran-safe)
    no = r'--'

    def cell(key, metric, fmt):
        if key in results and metric in results[key] and results[key][metric] == results[key][metric]:
            return fmt.format(results[key][metric])
        return no

    lines = []
    lines.append('% Auto-generated by exp6_R1_4_ablation_training.py')
    lines.append('% FIND the current ablation table in the manuscript and REPLACE it with this,')
    lines.append('% or paste this near the ablation discussion (Reviewer 1, comment 4).')
    lines.append('% Requires \\usepackage{booktabs} and \\usepackage{amssymb}.')
    lines.append('\\begin{table*}[t]')
    lines.append('\\centering')
    lines.append('\\footnotesize')
    lines.append('\\caption{Component ablation on the EPHNOGRAM held-out subjects. '
                 f'All variants share the training protocol ({EPOCHS} epochs, Adam, '
                 f'batch {BATCH_SIZE}, L1 weight 100). The harness reports blind test scores '
                 'without target-std rescaling. PSNR is in dB; the R-peak amplitude '
                 'error is in z-score units; the temporal F1 uses a plus/minus '
                 '50\\,ms window.}')
    lines.append('\\label{tab:exp6_ablation}')
    lines.append('\\begin{tabular}{lccccccccccc}')
    lines.append('\\toprule')
    lines.append('Variant & MHSA & MSDC & Polarity & Output & Adv. & '
                 'PSNR & SSIM & Corr. & R-peak amp. err. & Temporal F1 & Params (M) \\\\')
    lines.append('\\midrule')

    for key in row_order:
        cfg = VARIANTS.get(key, {})
        mhsa = yes if cfg.get('use_attention') else no
        msdc = yes if cfg.get('use_dilated') else no
        pol = yes if cfg.get('polarity') else no
        adv = yes if cfg.get('adversarial') else no
        out = cfg.get('output', '--')
        params = cfg.get('params_m', float('nan'))
        params_s = f'{params:.2f}' if params == params else no
        lines.append(
            f'{display.get(key, key)} & {mhsa} & {msdc} & {pol} & {out} & {adv} & '
            f'{cell(key, "psnr", "{:.2f}")} & '
            f'{cell(key, "ssim", "{:.3f}")} & '
            f'{cell(key, "corr", "{:.3f}")} & '
            f'{cell(key, "rpeak_amp_err", "{:.3f}")} & '
            f'{cell(key, "temporal_f1_50", "{:.2f}")} & '
            f'{params_s} \\\\')

    lines.append('\\bottomrule')
    lines.append('\\end{tabular}')
    lines.append('\\end{table*}')

    tex = '\n'.join(lines) + '\n'
    path = os.path.join(OUT, 'table_exp6_ablation.tex')
    with open(path, 'w') as f:
        f.write(tex)
    print('wrote', path)
    print(tex)
    return path


emit_latex_table(all_results)